# Large-Scale Protein Feature Enrichment

Enriches **Davis**, **KIBA**, and **BindingDB-KD** DTI parquet datasets with UniProt metadata and sequence descriptors (AAC / PAAC / CTD).

- Raw inputs: `data/raw/{dataset}.parquet`
- Test run (10 rows): `data/testrun/`
- Full run: `data/processed/`
- Davis `Target_ID` gene symbols are mapped to UniProt accessions before fetch.


In [1]:
%pip install -q requests pandas numpy tqdm propy3 biopython pyarrow


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


## Core pipeline

UniProt fetch/cache, metadata summarization, AAC/PAAC/CTD descriptors, Davis gene-symbol mapping, and dataset enrichment.


In [3]:
from __future__ import annotations



import json

import shutil

import time

from pathlib import Path

from typing import Any, Dict, List, Optional, Tuple



import numpy as np

import pandas as pd

import requests

from Bio.SeqUtils.ProtParam import ProteinAnalysis

from tqdm import tqdm



# --- Paths (overridden by notebook init) ---

ROOT = Path.cwd()

DATA_DIR = ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

LEGACY_RAW_DIR = ROOT / "raw"

CACHE_DIR = DATA_DIR / "cache"

UNIPROT_JSON_CACHE = CACHE_DIR / "uniprot_json"

DAVIS_MAP_CACHE = CACHE_DIR / "davis_target_to_uniprot.tsv"



UNIPROT_REST_BASE = "https://rest.uniprot.org/uniprotkb"

UNIPROT_SEARCH_BASE = "https://rest.uniprot.org/uniprotkb/search"



DATASETS = ["davis", "kiba", "bindingdb_kd"]



REQUEST_DELAY_S = 0.35

FORCE_REFETCH = False





def ensure_data_dirs() -> None:

    for d in [RAW_DIR, CACHE_DIR, UNIPROT_JSON_CACHE, DATA_DIR / "testrun", DATA_DIR / "processed"]:

        d.mkdir(parents=True, exist_ok=True)

    for name in DATASETS:

        dst = RAW_DIR / f"{name}.parquet"

        if dst.exists():

            continue

        src = LEGACY_RAW_DIR / f"{name}.parquet"

        if src.exists():

            shutil.copy2(src, dst)





# --- UniProt fetch ---





def fetch_uniprot_json(

    uniprot_id: str, *, timeout_s: int = 30, max_retries: int = 3

) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:

    url = f"{UNIPROT_REST_BASE}/{uniprot_id}.json"

    last_err: Optional[str] = None

    for attempt in range(1, max_retries + 1):

        try:

            resp = requests.get(

                url, timeout=timeout_s, headers={"User-Agent": "ProteinEnrichment/1.0"}

            )

            if resp.status_code == 200:

                try:

                    return resp.json(), None

                except Exception as e:

                    return None, f"json_decode_error: {e}"

            if resp.status_code in (404, 400):

                return None, f"http_{resp.status_code}"

            last_err = f"http_{resp.status_code}"

        except requests.RequestException as e:

            last_err = f"network_error: {e}"

        if attempt < max_retries:

            time.sleep(0.6 * (2 ** (attempt - 1)))

    return None, last_err or "unknown_error"





def _cache_path_for_id(uniprot_id: str) -> Path:

    safe = uniprot_id.replace("/", "_").replace("\\", "_")

    return UNIPROT_JSON_CACHE / f"{safe}.json"





def get_uniprot_record_cached(

    uniprot_id: str, *, force_refetch: bool = False

) -> Tuple[Optional[Dict[str, Any]], Optional[str], bool]:

    cache_path = _cache_path_for_id(uniprot_id)

    if not force_refetch and cache_path.exists():

        try:

            return json.loads(cache_path.read_text(encoding="utf-8")), None, True

        except Exception as e:

            record, err = fetch_uniprot_json(uniprot_id)

            if record is not None:

                cache_path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")

                return record, None, False

            return None, f"cache_read_error_then_{err or e}", False



    record, err = fetch_uniprot_json(uniprot_id)

    if record is not None:

        cache_path.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")

        return record, None, False

    return None, err, False





# --- Summarize ---





def _get(d: Any, path: List[Any], default=None):

    cur = d

    for p in path:

        try:

            if isinstance(p, int):

                cur = cur[p]

            else:

                cur = cur.get(p)

        except Exception:

            return default

        if cur is None:

            return default

    return cur





def _extract_crossref_ids(record: Dict[str, Any], db: str) -> List[str]:

    out: List[str] = []

    for x in record.get("uniProtKBCrossReferences", []) or []:

        if x.get("database") == db:

            pid = x.get("id")

            if pid:

                out.append(pid)

    return out





def _extract_comment_texts(record: Dict[str, Any], comment_type: str) -> List[str]:

    texts: List[str] = []

    for c in record.get("comments", []) or []:

        if c.get("commentType") != comment_type:

            continue

        if "texts" in c:

            for t in c.get("texts", []) or []:

                v = t.get("value") if isinstance(t, dict) else None

                if v:

                    texts.append(v)

        if "text" in c and isinstance(c.get("text"), dict):

            v = c["text"].get("value")

            if v:

                texts.append(v)

        if "subcellularLocations" in c:

            for sl in c.get("subcellularLocations", []) or []:

                loc = _get(sl, ["location", "value"]) or _get(sl, ["location", "id"])

                if loc:

                    texts.append(str(loc))

    return texts





def summarize_uniprot_record(uniprot_id: str, record: Dict[str, Any]) -> Dict[str, Any]:

    seq = _get(record, ["sequence", "value"], "") or ""

    seq_len = _get(record, ["sequence", "length"], None)

    checksum = _get(record, ["sequence", "checksum"], None)



    mw = None

    if seq:

        try:

            mw = float(ProteinAnalysis(seq).molecular_weight())

        except Exception:

            mw = None



    rec_name = _get(record, ["proteinDescription", "recommendedName", "fullName", "value"], None)

    ec_numbers: List[str] = []

    for ec in _get(record, ["proteinDescription", "recommendedName", "ecNumbers"], []) or []:

        v = ec.get("value") if isinstance(ec, dict) else None

        if v:

            ec_numbers.append(v)



    go_terms: List[str] = []

    for go in record.get("uniProtKBCrossReferences", []) or []:

        if go.get("database") == "GO":

            gid = go.get("id")

            if gid:

                go_terms.append(gid)



    keywords: List[str] = []

    for kw in record.get("keywords", []) or []:

        kid = kw.get("id") or kw.get("value")

        if kid:

            keywords.append(str(kid))



    reactome_ids = _extract_crossref_ids(record, "Reactome")

    ft = record.get("features", []) or []

    ft_types = [f.get("type") for f in ft if isinstance(f, dict)]



    def _count(t: str) -> int:

        return int(sum(1 for x in ft_types if x == t))



    subcell_texts = _extract_comment_texts(record, "SUBCELLULAR LOCATION")

    function_texts = _extract_comment_texts(record, "FUNCTION")

    pathway_texts = _extract_comment_texts(record, "PATHWAY")

    enzyme_reg_texts = _extract_comment_texts(record, "ENZYME REGULATION")

    tissue_texts = _extract_comment_texts(record, "TISSUE SPECIFICITY")

    dev_texts = _extract_comment_texts(record, "DEVELOPMENTAL STAGE")

    pdb_ids = _extract_crossref_ids(record, "PDB")



    isoform_count = 0

    for c in _get(record, ["comments"], []) or []:

        if isinstance(c, dict) and c.get("commentType") == "ALTERNATIVE PRODUCTS":

            isoform_count += len(c.get("isoforms", []) or [])



    return {

        "uniprot_id": uniprot_id,

        "sequence": seq,

        "sequence_length": seq_len if seq_len is not None else (len(seq) if seq else None),

        "sequence_checksum": checksum,

        "molecular_weight": mw,

        "recommended_name": rec_name,

        "ec_numbers": ";".join(sorted(set(ec_numbers))) if ec_numbers else None,

        "go_terms": ";".join(sorted(set(go_terms))) if go_terms else None,

        "keywords": ";".join(sorted(set(keywords))) if keywords else None,

        "reactome": ";".join(sorted(set(reactome_ids))) if reactome_ids else None,

        "subcellular_location": ";".join(sorted(set(subcell_texts))) if subcell_texts else None,

        "function": " ".join(function_texts) if function_texts else None,

        "pathway": " ".join(pathway_texts) if pathway_texts else None,

        "enzyme_regulation": " ".join(enzyme_reg_texts) if enzyme_reg_texts else None,

        "tissue_specificity": " ".join(tissue_texts) if tissue_texts else None,

        "developmental_stage": " ".join(dev_texts) if dev_texts else None,

        "isoform_count": isoform_count,

        "ft_transmem_count": _count("TRANSMEM"),

        "ft_topo_dom_count": _count("TOPO_DOM"),

        "ft_domain_count": _count("DOMAIN"),

        "ft_region_count": _count("REGION"),

        "ft_binding_count": _count("BINDING"),

        "ft_ptm_count": _count("MOD_RES") + _count("CARBOHYD"),

        "ft_variant_count": _count("VARIANT"),

        "ft_mutagen_count": _count("MUTAGEN"),

        "pdb_count": len(pdb_ids),

        "pdb_ids": ";".join(sorted(set(pdb_ids))) if pdb_ids else None,

    }





# --- Descriptors ---



AA20 = list("ACDEFGHIKLMNPQRSTVWY")



try:

    from propy import AAComposition

    from propy import CTD as PropyCTD

    from propy import PseudoAAC



    _HAVE_PROPY = True

except Exception:

    _HAVE_PROPY = False





def _aac_fallback(seq: str) -> Dict[str, float]:

    seq = seq.upper()

    n = len(seq)

    if n == 0:

        return {f"aac_{aa}": np.nan for aa in AA20}

    counts = {aa: 0 for aa in AA20}

    for ch in seq:

        if ch in counts:

            counts[ch] += 1

    return {f"aac_{aa}": counts[aa] / n for aa in AA20}





def _paac_fallback(seq: str, lambda_: int = 10, weight: float = 0.05) -> Dict[str, float]:

    aac = _aac_fallback(seq)

    seq = seq.upper()

    dipep_keys = ["AA", "AC", "CA", "CC", "GG", "PP", "RR", "SS", "TT", "VV"]

    dipep = {f"paac_dipep_{k}": 0.0 for k in dipep_keys}

    if len(seq) >= 2:

        total = len(seq) - 1

        for i in range(total):

            k = seq[i : i + 2]

            if k in dipep_keys:

                dipep[f"paac_dipep_{k}"] += 1.0

        for k in dipep_keys:

            dipep[f"paac_dipep_{k}"] /= total

    paac = {f"paac_{k.replace('aac_', '')}": float(v) for k, v in aac.items()}

    paac.update(dipep)

    paac["paac_lambda"] = float(lambda_)

    paac["paac_weight"] = float(weight)

    return paac





def _ctd_fallback(seq: str) -> Dict[str, float]:

    seq = "".join([c for c in seq.upper() if c.isalpha()])

    if not seq:

        return {"ctd_hydro_C1": np.nan, "ctd_hydro_C2": np.nan, "ctd_hydro_C3": np.nan}

    g1, g2, g3 = set("RKEDQN"), set("GASTPHY"), set("CLVIMFW")

    groups = []

    for aa in seq:

        if aa in g1:

            groups.append(1)

        elif aa in g2:

            groups.append(2)

        elif aa in g3:

            groups.append(3)

    if not groups:

        return {"ctd_hydro_C1": np.nan, "ctd_hydro_C2": np.nan, "ctd_hydro_C3": np.nan}

    n = len(groups)

    return {

        "ctd_hydro_C1": float(sum(1 for x in groups if x == 1) / n),

        "ctd_hydro_C2": float(sum(1 for x in groups if x == 2) / n),

        "ctd_hydro_C3": float(sum(1 for x in groups if x == 3) / n),

    }





def compute_descriptors(uniprot_id: str, sequence: str) -> Dict[str, float]:

    if not sequence:

        return {"uniprot_id": uniprot_id}

    features: Dict[str, float] = {"uniprot_id": uniprot_id}

    if _HAVE_PROPY:

        try:

            aac = AAComposition.CalculateAAComposition(sequence)

            for k, v in aac.items():

                features[f"aac_{k}"] = float(v)

        except Exception:

            features.update(_aac_fallback(sequence))

        try:

            paac = PseudoAAC.GetAPseudoAAC(sequence, lamda=10, weight=0.05)

            for k, v in paac.items():

                features[f"paac_{k}"] = float(v)

        except Exception:

            features.update(_paac_fallback(sequence))

        try:

            ctd = PropyCTD.CalculateCTD(sequence)

            for k, v in ctd.items():

                features[f"ctd_{k}"] = float(v)

        except Exception:

            features.update(_ctd_fallback(sequence))

        return features

    features.update(_aac_fallback(sequence))

    features.update(_paac_fallback(sequence))

    features.update(_ctd_fallback(sequence))

    return features





def build_protein_feature_df(

    uniprot_ids: List[str],

    *,

    force_refetch: bool = False,

    request_delay_s: float = 0.35,

) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """Return (protein_feature_df, failures_df)."""

    failures: List[Dict[str, Any]] = []

    metadata_rows: List[Dict[str, Any]] = []

    records: Dict[str, Dict[str, Any]] = {}



    for uid in tqdm(uniprot_ids, desc="UniProt fetch"):

        rec, err, _ = get_uniprot_record_cached(uid, force_refetch=force_refetch)

        if rec is None:

            failures.append(

                {"uniprot_id": uid, "stage": "fetch", "error": err}

            )

            continue

        records[uid] = rec

        if request_delay_s > 0:

            time.sleep(request_delay_s)



    for uid, rec in records.items():

        try:

            metadata_rows.append(summarize_uniprot_record(uid, rec))

        except Exception as e:

            failures.append({"uniprot_id": uid, "stage": "summarize", "error": str(e)})



    if not metadata_rows:

        return pd.DataFrame(), pd.DataFrame(failures)



    metadata_df = pd.DataFrame.from_records(metadata_rows).set_index("uniprot_id")

    feature_rows: List[Dict[str, Any]] = []

    for uid in metadata_df.index.tolist():

        seq = metadata_df.loc[uid, "sequence"]

        try:

            feature_rows.append(compute_descriptors(uid, seq))

        except Exception as e:

            failures.append({"uniprot_id": uid, "stage": "descriptors", "error": str(e)})



    features_df = pd.DataFrame.from_records(feature_rows).set_index("uniprot_id")

    protein_feature_df = metadata_df.drop(columns=["sequence"]).join(features_df, how="outer")

    failures_df = pd.DataFrame(failures)

    return protein_feature_df, failures_df





# --- Davis mapping ---





def map_target_to_uniprot(target: str, *, request_delay_s: float = 0.35) -> Optional[str]:

    t = str(target).rstrip("p")

    q1 = f"gene:{t} AND organism_id:9606"

    params = {"query": q1, "fields": "accession", "format": "tsv"}

    r = requests.get(UNIPROT_SEARCH_BASE, params=params, timeout=30)

    if request_delay_s > 0:

        time.sleep(request_delay_s)

    if r.status_code == 200 and len(r.text.splitlines()) > 1:

        return r.text.splitlines()[1].strip()

    q2 = f"entry_name:{t.lower()}"

    params["query"] = q2

    r2 = requests.get(UNIPROT_SEARCH_BASE, params=params, timeout=30)

    if request_delay_s > 0:

        time.sleep(request_delay_s)

    if r2.status_code == 200 and len(r2.text.splitlines()) > 1:

        return r2.text.splitlines()[1].strip()

    return None





def load_davis_mapping_cache() -> pd.DataFrame:

    if DAVIS_MAP_CACHE.exists():

        return pd.read_csv(DAVIS_MAP_CACHE, sep="\t")

    return pd.DataFrame(columns=["Target_ID", "uniprot_id", "map_error"])





def save_davis_mapping_cache(df: pd.DataFrame) -> None:

    DAVIS_MAP_CACHE.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(DAVIS_MAP_CACHE, sep="\t", index=False)





def resolve_davis_uniprot_ids(

    targets: List[str], *, request_delay_s: float = 0.35

) -> pd.DataFrame:

    cache = load_davis_mapping_cache()

    known = set(cache["Target_ID"].astype(str)) if len(cache) else set()

    rows = []

    for tgt in targets:

        tgt_s = str(tgt)

        if tgt_s in known:

            continue

        acc = map_target_to_uniprot(tgt_s, request_delay_s=request_delay_s)

        rows.append(

            {

                "Target_ID": tgt_s,

                "uniprot_id": acc,

                "map_error": None if acc else "mapping_failed",

            }

        )

    if rows:

        cache = pd.concat([cache, pd.DataFrame(rows)], ignore_index=True)

        cache = cache.drop_duplicates(subset=["Target_ID"], keep="last")

        save_davis_mapping_cache(cache)

    return cache[cache["Target_ID"].astype(str).isin([str(t) for t in targets])]





def resolve_uniprot_ids(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:

    out = df.copy()

    if dataset_name == "davis":

        targets = out["Target_ID"].drop_duplicates().tolist()

        mapping = resolve_davis_uniprot_ids(targets, request_delay_s=REQUEST_DELAY_S)

        out = out.merge(mapping, on="Target_ID", how="left")

        if "map_error" not in out.columns:

            out["map_error"] = None

    else:

        out["uniprot_id"] = out["Target_ID"]

        out["map_error"] = np.where(out["Target_ID"].isna(), "null_target_id", None)

    return out





def enrich_dataset(

    dataset_name: str,

    *,

    test_run: bool = False,

    n_test_rows: int = 10,

    force_refetch: bool = False,

    request_delay_s: float = 0.35,

) -> Dict[str, Any]:

    ensure_data_dirs()

    input_path = RAW_DIR / f"{dataset_name}.parquet"

    out_dir = DATA_DIR / ("testrun" if test_run else "processed")

    out_dir.mkdir(parents=True, exist_ok=True)



    raw_df = pd.read_parquet(input_path)

    if test_run:

        work_df = raw_df.head(n_test_rows).copy()

    else:

        work_df = raw_df.copy()



    work_df = resolve_uniprot_ids(work_df, dataset_name)



    feature_cache = CACHE_DIR / f"protein_features_{dataset_name}.parquet"

    failures_all: List[Dict[str, Any]] = []



    ids = (

        work_df["uniprot_id"]

        .dropna()

        .astype(str)

        .str.strip()

        .replace("", np.nan)

        .dropna()

        .unique()

        .tolist()

    )



    use_feature_cache = (

        feature_cache.exists() and not force_refetch and not test_run

    )

    if use_feature_cache:

        protein_feature_df = pd.read_parquet(feature_cache).set_index("uniprot_id")

        fetch_failures = pd.DataFrame()

    else:

        protein_feature_df, fetch_failures = build_protein_feature_df(

            ids, force_refetch=force_refetch, request_delay_s=request_delay_s

        )

        if not test_run and len(protein_feature_df):

            protein_feature_df.reset_index().to_parquet(feature_cache, index=False)



    if len(fetch_failures):

        ff = fetch_failures.copy()

        ff["dataset"] = dataset_name

        failures_all.extend(ff.to_dict("records"))



    if dataset_name == "davis":

        unmapped = work_df[work_df["uniprot_id"].isna()]

        for _, row in unmapped.iterrows():

            failures_all.append(

                {

                    "dataset": dataset_name,

                    "original_target_id": row["Target_ID"],

                    "uniprot_id": None,

                    "stage": "map",

                    "error": row.get("map_error") or "mapping_failed",

                }

            )



    null_targets = work_df[work_df["Target_ID"].isna()]

    for _ in range(len(null_targets)):

        failures_all.append(

            {

                "dataset": dataset_name,

                "original_target_id": None,

                "uniprot_id": None,

                "stage": "map",

                "error": "null_target_id",

            }

        )



    feat_reset = protein_feature_df.reset_index()

    enriched = work_df.merge(feat_reset, on="uniprot_id", how="left", suffixes=("", "_feat"))



    out_path = out_dir / f"{dataset_name}_enriched.parquet"

    enriched.to_parquet(out_path, index=False)



    fail_path = out_dir / f"{dataset_name}_fetch_failures.parquet"

    failures_df = pd.DataFrame(failures_all)

    if len(failures_df):

        failures_df.to_parquet(fail_path, index=False)

    elif fail_path.exists():

        fail_path.unlink()



    n_feat_cols = len([c for c in enriched.columns if c.startswith(("aac_", "paac_", "ctd_"))])

    return {

        "dataset": dataset_name,

        "rows": len(enriched),

        "columns": len(enriched.columns),

        "feature_descriptor_cols": n_feat_cols,

        "unique_uniprot_fetched": len(protein_feature_df),

        "output": str(out_path),

        "failures": len(failures_df),

    }



c:\Users\Abdullah\anaconda3\envs\dti_research\lib\site-packages\propy\PseudoAAC.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [4]:
# --- Configuration (set before running enrichment) ---
TEST_RUN = False          # True: first 10 rows -> data/testrun/
FULL_RUN = True         # True: full datasets -> data/processed/
N_TEST_ROWS = 10
FORCE_REFETCH = False
REQUEST_DELAY_S = 0.35


## Test run (first 10 rows per dataset)


In [5]:
ensure_data_dirs()

test_summaries = []
if TEST_RUN:
    for name in DATASETS:
        print(f"\n=== TEST RUN: {name} ===")
        summary = enrich_dataset(
            name,
            test_run=True,
            n_test_rows=N_TEST_ROWS,
            force_refetch=FORCE_REFETCH,
            request_delay_s=REQUEST_DELAY_S,
        )
        test_summaries.append(summary)
        print(summary)
else:
    print("Set TEST_RUN = True to run smoke test.")


Set TEST_RUN = True to run smoke test.


## Full-scale run


In [6]:
full_summaries = []
if FULL_RUN:
    for name in DATASETS:
        print(f"\n=== FULL RUN: {name} ===")
        summary = enrich_dataset(
            name,
            test_run=False,
            force_refetch=FORCE_REFETCH,
            request_delay_s=REQUEST_DELAY_S,
        )
        full_summaries.append(summary)
        print(summary)
else:
    print("Set FULL_RUN = True after test run succeeds.")



=== FULL RUN: davis ===


UniProt fetch: 100%|██████████| 342/342 [08:20<00:00,  1.46s/it]


{'dataset': 'davis', 'rows': 25772, 'columns': 239, 'feature_descriptor_cols': 207, 'unique_uniprot_fetched': 342, 'output': 'd:\\Thesis\\CodeWork\\AllData\\data\\processed\\davis_enriched.parquet', 'failures': 2516}

=== FULL RUN: kiba ===


UniProt fetch: 100%|██████████| 229/229 [02:32<00:00,  1.50it/s]


{'dataset': 'kiba', 'rows': 117657, 'columns': 239, 'feature_descriptor_cols': 207, 'unique_uniprot_fetched': 229, 'output': 'd:\\Thesis\\CodeWork\\AllData\\data\\processed\\kiba_enriched.parquet', 'failures': 0}

=== FULL RUN: bindingdb_kd ===


UniProt fetch: 100%|██████████| 1090/1090 [20:15<00:00,  1.12s/it]


{'dataset': 'bindingdb_kd', 'rows': 52274, 'columns': 271, 'feature_descriptor_cols': 239, 'unique_uniprot_fetched': 1089, 'output': 'd:\\Thesis\\CodeWork\\AllData\\data\\processed\\bindingdb_kd_enriched.parquet', 'failures': 4334}


## Validation


In [7]:
out_dir = DATA_DIR / ("testrun" if TEST_RUN and not FULL_RUN else "processed")
if TEST_RUN and not FULL_RUN:
    check_dir = DATA_DIR / "testrun"
elif FULL_RUN:
    check_dir = DATA_DIR / "processed"
else:
    check_dir = DATA_DIR / "testrun"

for name in DATASETS:
    p = check_dir / f"{name}_enriched.parquet"
    if not p.exists():
        print(f"Missing: {p}")
        continue
    df = pd.read_parquet(p)
    n_desc = len([c for c in df.columns if c.startswith(("aac_", "paac_", "ctd_"))])
    print(f"{name}: shape={df.shape}, descriptor_cols={n_desc}")
    display(df.head(2))
    if name == "davis" and "uniprot_id" in df.columns:
        print(df[["Target_ID", "uniprot_id"]].drop_duplicates().head(5))


davis: shape=(25772, 239), descriptor_cols=207


,Drug_ID,drug_smiles,Target_ID,target_sequence,affinity_label,uniprot_id,map_error,sequence_length,sequence_checksum,molecular_weight,recommended_name,ec_numbers,go_terms,keywords,reactome,subcellular_location,function,pathway,enzyme_regulation,tissue_specificity,developmental_stage,isoform_count,ft_transmem_count,ft_topo_dom_count,ft_domain_count,ft_region_count,ft_binding_count,ft_ptm_count,ft_variant_count,ft_mutagen_count,pdb_count,pdb_ids,aac_A,aac_R,aac_N,aac_D,aac_C,aac_E,aac_Q,aac_G,...,ctd__PolarityD2001,ctd__PolarityD2025,ctd__PolarityD2050,ctd__PolarityD2075,ctd__PolarityD2100,ctd__PolarityD3001,ctd__PolarityD3025,ctd__PolarityD3050,ctd__PolarityD3075,ctd__PolarityD3100,ctd__NormalizedVDWVD1001,ctd__NormalizedVDWVD1025,ctd__NormalizedVDWVD1050,ctd__NormalizedVDWVD1075,ctd__NormalizedVDWVD1100,ctd__NormalizedVDWVD2001,ctd__NormalizedVDWVD2025,ctd__NormalizedVDWVD2050,ctd__NormalizedVDWVD2075,ctd__NormalizedVDWVD2100,ctd__NormalizedVDWVD3001,ctd__NormalizedVDWVD3025,ctd__NormalizedVDWVD3050,ctd__NormalizedVDWVD3075,ctd__NormalizedVDWVD3100,ctd__HydrophobicityD1001,ctd__HydrophobicityD1025,ctd__HydrophobicityD1050,ctd__HydrophobicityD1075,ctd__HydrophobicityD1100,ctd__HydrophobicityD2001,ctd__HydrophobicityD2025,ctd__HydrophobicityD2050,ctd__HydrophobicityD2075,ctd__HydrophobicityD2100,ctd__HydrophobicityD3001,ctd__HydrophobicityD3025,ctd__HydrophobicityD3050,ctd__HydrophobicityD3075,ctd__HydrophobicityD3100
0,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,AAK1,MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...,43.0,Q2M2I8,None,961.0,None,103883.7165,AP2-associated protein kinase 1,2.7.11.1,GO:0004674;GO:0005112;GO:0005524;GO:0005829;GO...,KW-0002;KW-0007;KW-0025;KW-0067;KW-0168;KW-025...,R-HSA-8856825;R-HSA-8856828,"Cell membrane;Membrane, clathrin-coated pit;Pr...",Regulates clathrin-mediated endocytosis by pho...,None,None,"Detected in brain, heart and liver. Isoform 1 ...",None,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,4WSQ;5L4Q;5TE0;8GMC;8GMD;9F6S;9F8T;9QB5,8.221,3.330,3.434,4.683,1.353,4.475,11.655,6.764,...,0.728,34.964,57.024,77.211,98.959,0.208,23.829,49.428,71.592,99.896,0.624,29.969,54.214,78.044,99.896,1.041,23.205,50.156,73.881,100.000,0.104,18.106,32.882,66.701,98.855,0.208,24.662,49.532,71.904,99.896,0.728,31.426,53.174,76.067,98.959,0.104,16.857,37.149,77.732,100.000
1,11314340,Cc1[nH]nc2ccc(-c3cncc(OCC(N)Cc4ccccc4)c3)cc12,ABL1p,PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...,10000.0,P00519,None,1130.0,None,122871.0844,Tyrosine-protein kinase ABL1,2.7.10.2,GO:0000278;GO:0000287;GO:0000400;GO:0000405;GO...,KW-0002;KW-0007;KW-0025;KW-0053;KW-0067;KW-007...,R-HSA-2029482;R-HSA-428890;R-HSA-525793;R-HSA-...,"Cytoplasm, cytoskeleton;Mitochondrion;Nucleus;...",Non-receptor tyrosine-protein kinase that play...,None,None,Widely expressed,None,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,84.0,1AB2;1AWO;1BBZ;1JU5;1OPL;1ZZP;2ABL;2E2B;2F4J;2...,8.584,5.664,3.540,3.717,1.239,7.965,2.920,7.788,...,0.885,32.920,59.381,79.469,99.558,0.265,24.425,48.584,70.442,100.000,0.442,32.124,58.142,78.584,99.646,0.177,21.947,42.389,71.858,99.912,0.088,23.097,42.743,67.168,100.000,0.265,24.956,49.381,70.796,100.000,0.885,27.345,56.814,77.257,99.558,0.088,22.655,39.204,69.558,99.823


  Target_ID uniprot_id
0      AAK1     Q2M2I8
1     ABL1p     P00519
2      ABL2     P42684
3     ACVR1     Q04771
4    ACVR1B     P36896
kiba: shape=(117657, 239), descriptor_cols=207


,Drug_ID,drug_smiles,Target_ID,target_sequence,affinity_label,uniprot_id,map_error,sequence_length,sequence_checksum,molecular_weight,recommended_name,ec_numbers,go_terms,keywords,reactome,subcellular_location,function,pathway,enzyme_regulation,tissue_specificity,developmental_stage,isoform_count,ft_transmem_count,ft_topo_dom_count,ft_domain_count,ft_region_count,ft_binding_count,ft_ptm_count,ft_variant_count,ft_mutagen_count,pdb_count,pdb_ids,aac_A,aac_R,aac_N,aac_D,aac_C,aac_E,aac_Q,aac_G,...,ctd__PolarityD2001,ctd__PolarityD2025,ctd__PolarityD2050,ctd__PolarityD2075,ctd__PolarityD2100,ctd__PolarityD3001,ctd__PolarityD3025,ctd__PolarityD3050,ctd__PolarityD3075,ctd__PolarityD3100,ctd__NormalizedVDWVD1001,ctd__NormalizedVDWVD1025,ctd__NormalizedVDWVD1050,ctd__NormalizedVDWVD1075,ctd__NormalizedVDWVD1100,ctd__NormalizedVDWVD2001,ctd__NormalizedVDWVD2025,ctd__NormalizedVDWVD2050,ctd__NormalizedVDWVD2075,ctd__NormalizedVDWVD2100,ctd__NormalizedVDWVD3001,ctd__NormalizedVDWVD3025,ctd__NormalizedVDWVD3050,ctd__NormalizedVDWVD3075,ctd__NormalizedVDWVD3100,ctd__HydrophobicityD1001,ctd__HydrophobicityD1025,ctd__HydrophobicityD1050,ctd__HydrophobicityD1075,ctd__HydrophobicityD1100,ctd__HydrophobicityD2001,ctd__HydrophobicityD2025,ctd__HydrophobicityD2050,ctd__HydrophobicityD2075,ctd__HydrophobicityD2100,ctd__HydrophobicityD3001,ctd__HydrophobicityD3025,ctd__HydrophobicityD3050,ctd__HydrophobicityD3075,ctd__HydrophobicityD3100
0,CHEMBL1087421,COc1cc2c(cc1Cl)C(c1ccc(Cl)c(Cl)c1)=NCC2,O00141,MTVKTEAAKGTLTYSRMRGMVAILIAFMKQRRMGLNDFIQKIANNS...,11.1,O00141,None,431,None,48941.7969,Serine/threonine-protein kinase Sgk1,2.7.11.1,GO:0001558;GO:0004674;GO:0004712;GO:0005246;GO...,KW-0002;KW-0025;KW-0053;KW-0067;KW-0256;KW-034...,R-HSA-1257604;R-HSA-2672351;R-HSA-6804757;R-HS...,Cell membrane;Cytoplasm;Endoplasmic reticulum ...,Serine/threonine-protein kinase which is invol...,None,None,Expressed in most tissues with highest levels ...,None,5,0,0,0,0,0,0,0,0,4,2R5T;3HDM;3HDN;7PUE,6.497,3.944,5.800,4.408,1.160,6.265,3.248,4.872,...,0.464,20.186,52.204,79.118,99.536,0.928,25.754,46.404,74.71,99.304,0.464,21.578,55.684,79.814,99.536,0.696,26.45,50.116,73.086,100.000,0.232,25.754,44.084,70.766,99.768,0.928,25.754,47.564,75.406,99.304,0.464,21.578,49.884,74.942,99.536,0.232,26.450,49.420,72.854,100.000
1,CHEMBL1087421,COc1cc2c(cc1Cl)C(c1ccc(Cl)c(Cl)c1)=NCC2,O14920,MSWSPSLTTQTCGAWEMKERLGTGGFGNVIRWHNQETGEQIAIKQC...,11.1,O14920,None,756,None,86562.9500,Inhibitor of nuclear factor kappa-B kinase sub...,2.7.11.10,GO:0002223;GO:0002479;GO:0002755;GO:0004672;GO...,KW-0002;KW-0007;KW-0025;KW-0067;KW-0225;KW-037...,R-HSA-1169091;R-HSA-1236974;R-HSA-168638;R-HSA...,Cytoplasm;Membrane raft;Nucleus,Serine kinase that plays an essential role in ...,None,None,"Highly expressed in heart, placenta, skeletal ...",None,4,0,0,0,0,0,0,0,0,5,3BRT;3BRV;4E3C;4KIK;8OMV,5.026,5.952,4.365,4.762,2.381,8.862,8.069,4.894,...,0.265,25.132,48.413,76.455,100.000,1.323,25.794,53.704,76.19,99.735,0.265,24.471,48.545,74.206,100.000,0.926,25.00,51.058,76.190,99.735,0.132,24.074,51.323,72.619,99.074,1.323,26.058,55.026,76.323,99.735,0.265,23.942,45.635,73.810,100.000,0.132,24.206,47.884,71.429,99.471


bindingdb_kd: shape=(52274, 271), descriptor_cols=239


,Drug_ID,drug_smiles,Target_ID,target_sequence,affinity_label,uniprot_id,map_error,sequence_length,sequence_checksum,molecular_weight,recommended_name,ec_numbers,go_terms,keywords,reactome,subcellular_location,function,pathway,enzyme_regulation,tissue_specificity,developmental_stage,isoform_count,ft_transmem_count,ft_topo_dom_count,ft_domain_count,ft_region_count,ft_binding_count,ft_ptm_count,ft_variant_count,ft_mutagen_count,pdb_count,pdb_ids,aac_A,aac_R,aac_N,aac_D,aac_C,aac_E,aac_Q,aac_G,...,ctd__HydrophobicityD2050,ctd__HydrophobicityD2075,ctd__HydrophobicityD2100,ctd__HydrophobicityD3001,ctd__HydrophobicityD3025,ctd__HydrophobicityD3050,ctd__HydrophobicityD3075,ctd__HydrophobicityD3100,paac_A,paac_C,paac_D,paac_E,paac_F,paac_G,paac_H,paac_I,paac_K,paac_L,paac_M,paac_N,paac_P,paac_Q,paac_R,paac_S,paac_T,paac_V,paac_W,paac_Y,paac_dipep_AA,paac_dipep_AC,paac_dipep_CA,paac_dipep_CC,paac_dipep_GG,paac_dipep_PP,paac_dipep_RR,paac_dipep_SS,paac_dipep_TT,paac_dipep_VV,paac_lambda,paac_weight
0,444607.0,Cc1ccc(CNS(=O)(=O)c2ccc(S(N)(=O)=O)s2)cc1,P00918,MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKP...,0.46,P00918,None,260.0,None,29245.6681,Carbonic anhydrase 2,4.2.1.1,GO:0002009;GO:0004064;GO:0004089;GO:0005737;GO...,KW-0002;KW-0007;KW-0225;KW-0456;KW-0472;KW-047...,R-HSA-1237044;R-HSA-1247673;R-HSA-1475029;R-HS...,Cell membrane;Cytoplasm,Catalyzes the reversible hydration of carbon d...,None,None,None,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1151.0,12CA;1A42;1AM6;1AVN;1BCD;1BIC;1BN1;1BN3;1BN4;1...,5.0,2.692,3.846,7.308,0.385,5.0,4.231,8.462,...,43.846,69.231,99.231,0.385,30.385,56.154,78.846,99.615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4316.0,COc1ccc(CNS(=O)(=O)c2ccc(S(N)(=O)=O)s2)cc1,P00918,MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKP...,0.49,P00918,None,260.0,None,29245.6681,Carbonic anhydrase 2,4.2.1.1,GO:0002009;GO:0004064;GO:0004089;GO:0005737;GO...,KW-0002;KW-0007;KW-0225;KW-0456;KW-0472;KW-047...,R-HSA-1237044;R-HSA-1247673;R-HSA-1475029;R-HS...,Cell membrane;Cytoplasm,Catalyzes the reversible hydration of carbon d...,None,None,None,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1151.0,12CA;1A42;1AM6;1AVN;1BCD;1BIC;1BN1;1BN3;1BN4;1...,5.0,2.692,3.846,7.308,0.385,5.0,4.231,8.462,...,43.846,69.231,99.231,0.385,30.385,56.154,78.846,99.615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
